# Two-minus sector amplitude $A_n$ — closed forms & verification

Run top-to-bottom. Uses `bg.py` (exact Gaussian-rational port of `OnShellBG.m`) and `closed_form.py`.

**Findings.** $A_n$ is purely imaginary, homogeneous of degree $2(n-2)$, **bounded and piecewise** (NOT a single rational function). Closed forms:
- principal regime (smallest $|\omega|$ is a $\sigma=-1$ leg): $A_n = i\,2^{n-1}\omega_1\omega_2(\min(\omega_1^2,\omega_2^2))^{n-3}$;
- complete $n=5$ (all chambers): $A_5 = i[P_0 + \sum_{\mu,j}P_{\mu j}|\omega_j^2-\omega_\mu^2|]$.

In [ ]:
from fractions import Fraction as Q
from bg import amp_two_minus
from closed_form import A_principal, A5_complete, softest_is_minus

# exact BG port reproduces Mathematica BGAmplitude
for n, free in [(5,[Q(2),Q(5,2),Q(3)]), (6,[Q(3,2),Q(2),Q(5,2),Q(3)])]:
    A,_,w = amp_two_minus(n, free)
    print(f'n={n} {free}:  A = {A.im} i')

In [ ]:
# (1) principal-regime formula, exact, n = 5,6,7
import random
for n in [5,6,7]:
    rng=random.Random(n); ok=tot=tries=0
    while tot < {5:60,6:20,7:5}[n] and tries < 40000:
        tries+=1
        free=[Q(rng.randint(-20,20),rng.randint(1,5)) for _ in range(n-2)]
        if any(x==0 for x in free): continue
        try: A,_,w=amp_two_minus(n,free)
        except Exception: continue
        if any(v==0 for v in w) or not softest_is_minus(w): continue
        tot+=1; ok += (A.im == A_principal(n,w))
    print(f'n={n}: principal formula exact on {ok}/{tot} random points (softest=minus)')

In [ ]:
# (2) complete n=5 formula, exact on ALL chambers
rng=random.Random(5); ok=tot=0
for _ in range(3000):
    free=[Q(rng.randint(-30,30),rng.randint(1,7)) for _ in range(3)]
    if any(x==0 for x in free): continue
    try: A,_,w=amp_two_minus(5,free)
    except Exception: continue
    if any(v==0 for v in w): continue
    tot+=1; ok += (A.im == A5_complete(w))
    if tot>=200: break
print(f'n=5 complete formula exact on {ok}/{tot} random points (all chambers)')

In [ ]:
# (3) proof: A_5 is two DIFFERENT rational functions on two open sets
rng=random.Random(11); a=ta=b=tb=0
for _ in range(40000):
    free=[Q(rng.randint(-20,20),rng.randint(1,5)) for _ in range(3)]
    if any(x==0 for x in free): continue
    try: A,_,w=amp_two_minus(5,free)
    except Exception: continue
    if any(v==0 for v in w): continue
    w2=[v*v for v in w]
    if w2.index(min(w2))>=2: continue
    if w[0]**2<w[1]**2: ta+=1; a+=(A.im==16*w[0]**5*w[1])
    else: tb+=1; b+=(A.im==16*w[0]*w[1]**5)
    if ta+tb>=300: break
print(f'|w1|<|w2|: A5==16 w1^5 w2 -> {a}/{ta};  |w2|<|w1|: A5==16 w1 w2^5 -> {b}/{tb}')
print('=> not a single rational function.')